In [1]:
# ============================================================
# Synthetic Disorder Step6 setup
#
# This notebook does NOT upload or read config.py.
# All synthetic Step6 settings are defined explicitly here
# to avoid confusion with the main ExpB SettingA1 pipeline.
#
# Input:
#   Complete synthetic disorder Step5 hidden-state .pt file
#   generated from the clean LRAB Step4 dataset.
#
# Output:
#   Step6 linear-probe results for implicit-transitive LRAB labels:
#       left_of, right_of, above, below
# ============================================================

import os
import sys
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

# ------------------------------------------------------------
# 1. Create project structure
# ------------------------------------------------------------

PILOT_ROOT = "/content/pilot_4"
DATA_DIR = os.path.join(PILOT_ROOT, "data")
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)

with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# ------------------------------------------------------------
# 2. Explicit synthetic disorder Step6 config
# ------------------------------------------------------------

# Logical experiment/data version used for Step6 output naming.
# The actual Step5 .pt filename contains "lrab_25000"; we read it directly below.
STEP4_OUTPUT_BASENAME = "pilot4_synthetic_disorder_lrab_2500"

# Actual upstream Step5 source basename as saved on disk.
STEP5_SOURCE_BASENAME = "pilot4_synthetic_disorder_lrab_25000"

SYNTHETIC_EXPERIMENT_NAME = f"{STEP4_OUTPUT_BASENAME}_step6"

EXPERIMENT_NAME = (
    "pilot4_synthetic_disorder_lrab_2500_"
    "implicit_transitive_probe"
)

RANDOM_SEED = 42

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
MODEL_TAG = "Qwen2_5_0_5B"

STEP6_INPUT_DIR = os.path.join(
    DATA_DIR,
    f"{STEP4_OUTPUT_BASENAME}_step6_input_hidden_states_qwen2_5_0_5b"
)

STEP6_OUTPUT_DIR = os.path.join(
    DATA_DIR,
    f"{STEP4_OUTPUT_BASENAME}_step6_outputs_implicit_transitive_lrab_qwen2_5_0_5b"
)

# Keep the original probe feature and label fields.
# This preserves the old Step6 probe logic.
FEATURE_KEY = "layer_diff_features"
LABEL_FIELD = "relation"

TRAIN_SCENES = [
    "FloorPlan1",
    "FloorPlan2",
    "FloorPlan3",
    "FloorPlan4",
]

TEST_SCENES = [
    "FloorPlan5",
    "FloorPlan6",
]

REQUIRE_EXPLICIT_SCENE_SPLIT = True

# Step6 only uses implicit-transitive examples.
ALLOWED_EVIDENCE_TYPES = {
    "implicit_transitive",
}

# ------------------------------------------------------------
# Probe label definition
# ------------------------------------------------------------
# This is the decisive definition of the Step6 probe task.
#
# The linear probe predicts:
#   y ∈ {left_of, right_of, above, below}
#
# Other relation labels may appear in Step4 text as distractors,
# but they are not used as probe labels.
# ------------------------------------------------------------

PROBE_RELATION_LABELS = {
    "left_of",
    "right_of",
    "above",
    "below",
}

ALLOWED_LABELS = set(PROBE_RELATION_LABELS)
EXPECTED_RELATION_LABELS = set(PROBE_RELATION_LABELS)

GROUP_BY_IMPLICIT_RULE = True

LOGREG_MAX_ITER = 5000
LOGREG_C = 1.0

PRINT_DATASET_SUMMARY = True
PRINT_LAYER_PROGRESS = True
PRINT_TOP_LAYERS = True
NUM_TOP_LAYERS_TO_PRINT = 10

# ------------------------------------------------------------
# 3. Prepare input/output directories
# ------------------------------------------------------------

os.makedirs(STEP6_INPUT_DIR, exist_ok=True)

# Clear old Step6 input files.
for old_path in Path(STEP6_INPUT_DIR).glob("*"):
    if old_path.is_file():
        old_path.unlink()
    elif old_path.is_dir():
        shutil.rmtree(old_path)

# Clear old Step6 outputs.
if os.path.exists(STEP6_OUTPUT_DIR):
    shutil.rmtree(STEP6_OUTPUT_DIR)

os.makedirs(STEP6_OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 4. Directly read the complete LRAB Step5 .pt from Google Drive
# ------------------------------------------------------------

step5_pt_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/"
    "expB_settingA1_6fps/setting_synthetic/"
    "data_synthetic_disorder_lrab_2500/"
    "step5_hidden_states_qwen2_5_0_5b_lrab_2500/"
    "step4T_pilot4_synthetic_disorder_step4_lrab_25000_"
    "step5_hidden_states_Qwen2_5_0_5B.pt"
)

DRIVE_STEP5_OUTPUT_DIR = step5_pt_path.parent

# Summary is optional. This tries the filename matching the actual Step5 source basename.
step5_summary_path = DRIVE_STEP5_OUTPUT_DIR / (
    f"{STEP5_SOURCE_BASENAME}_step5_summary_{MODEL_TAG}.json"
)

print("\nUsing clean LRAB Step5 hidden-state .pt from Google Drive:")
print(step5_pt_path)

if not step5_pt_path.exists():
    print("\nFiles currently visible in DRIVE_STEP5_OUTPUT_DIR:")
    if DRIVE_STEP5_OUTPUT_DIR.exists():
        for p in sorted(DRIVE_STEP5_OUTPUT_DIR.glob("*")):
            print("-", p.name)
    else:
        print("Directory does not exist:", DRIVE_STEP5_OUTPUT_DIR)

    raise FileNotFoundError(
        "Clean LRAB Step5 .pt not found.\n"
        "The Step6 script is now configured to read the full Step5 output path directly.\n"
        f"Expected path:\n{step5_pt_path}"
    )

# ------------------------------------------------------------
# 5. Copy Step5 .pt and summary into STEP6_INPUT_DIR
# ------------------------------------------------------------

target_pt_path = Path(STEP6_INPUT_DIR) / step5_pt_path.name

print("\nCopying Step5 .pt into STEP6_INPUT_DIR:")
print("from:", step5_pt_path)
print("to:  ", target_pt_path)
print("size MB:", round(step5_pt_path.stat().st_size / 1024 / 1024, 2))

shutil.copy2(str(step5_pt_path), str(target_pt_path))

if target_pt_path.stat().st_size != step5_pt_path.stat().st_size:
    raise RuntimeError(
        "Copied .pt file size does not match source file size.\n"
        f"source={step5_pt_path.stat().st_size}\n"
        f"target={target_pt_path.stat().st_size}"
    )

if step5_summary_path.exists():
    target_summary_path = Path(STEP6_INPUT_DIR) / step5_summary_path.name
    shutil.copy2(str(step5_summary_path), str(target_summary_path))
else:
    target_summary_path = None
    print("\nWarning: Step5 summary JSON not found:")
    print(step5_summary_path)
    print("This does not block Step6 because the .pt file is the required input.")

pt_files = sorted(Path(STEP6_INPUT_DIR).rglob("*.pt"))

if not pt_files:
    raise FileNotFoundError(
        f"No .pt files found after copying Step5 output into {STEP6_INPUT_DIR}"
    )

# ------------------------------------------------------------
# 6. Setup summary
# ------------------------------------------------------------

print("\nSynthetic disorder LRAB Step6 setup complete.")
print("STEP4_OUTPUT_BASENAME:", STEP4_OUTPUT_BASENAME)
print("STEP5_SOURCE_BASENAME:", STEP5_SOURCE_BASENAME)
print("SYNTHETIC_EXPERIMENT_NAME:", SYNTHETIC_EXPERIMENT_NAME)
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("MODEL_NAME:", MODEL_NAME)
print("MODEL_TAG:", MODEL_TAG)
print("STEP6_INPUT_DIR:", STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", STEP6_OUTPUT_DIR)
print("FEATURE_KEY:", FEATURE_KEY)
print("LABEL_FIELD:", LABEL_FIELD)
print("TRAIN_SCENES:", TRAIN_SCENES)
print("TEST_SCENES:", TEST_SCENES)
print("ALLOWED_EVIDENCE_TYPES:", sorted(ALLOWED_EVIDENCE_TYPES))
print("PROBE_RELATION_LABELS:", sorted(PROBE_RELATION_LABELS))
print("ALLOWED_LABELS:", sorted(ALLOWED_LABELS))
print("EXPECTED_RELATION_LABELS:", sorted(EXPECTED_RELATION_LABELS))
print("Clean LRAB Step5 .pt:", step5_pt_path)
print("Copied Step5 summary:", target_summary_path)
print("Number of .pt files found:", len(pt_files))

print("\nStep6 input .pt files:")
for p in pt_files[:10]:
    print("-", p, "|", round(p.stat().st_size / 1024 / 1024, 2), "MB")

Mounted at /content/drive

Using clean LRAB Step5 hidden-state .pt from Google Drive:
/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expB_settingA1_6fps/setting_synthetic/data_synthetic_disorder_lrab_2500/step5_hidden_states_qwen2_5_0_5b_lrab_2500/step4T_pilot4_synthetic_disorder_step4_lrab_25000_step5_hidden_states_Qwen2_5_0_5B.pt

Copying Step5 .pt into STEP6_INPUT_DIR:
from: /content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expB_settingA1_6fps/setting_synthetic/data_synthetic_disorder_lrab_2500/step5_hidden_states_qwen2_5_0_5b_lrab_2500/step4T_pilot4_synthetic_disorder_step4_lrab_25000_step5_hidden_states_Qwen2_5_0_5B.pt
to:   /content/pilot_4/data/pilot4_synthetic_disorder_lrab_2500_step6_input_hidden_states_qwen2_5_0_5b/step4T_pilot4_synthetic_disorder_step4_lrab_25000_step5_hidden_states_Qwen2_5_0_5B.pt
size MB: 2324.07

/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expB_settingA1_6fps/setting_synthetic/data_synthetic_disorder_lrab_2500/step5_hid

In [2]:
# ============================================================
# Imports and Step6 configuration check
#
# Current Step6 goal:
#   Decode clean LRAB implicit-transitive spatial relation labels
#   from synthetic Step5 hidden states.
#
# Main task:
#   X = hidden(probe_subject) - hidden(probe_object)
#   y = relation label
#
# Probe label space:
#   left_of, right_of, above, below
#
# Important:
#   Other relation labels may appear in the Step4 text only as
#   distractor relation sentences. They are not probed here.
# ============================================================

import os
import sys
import json
import random
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

os.makedirs(STEP6_OUTPUT_DIR, exist_ok=True)

print("Synthetic LRAB Step6 config loaded.")
print("STEP4_OUTPUT_BASENAME:", STEP4_OUTPUT_BASENAME)
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("MODEL_NAME:", MODEL_NAME)
print("MODEL_TAG:", MODEL_TAG)
print("STEP6_INPUT_DIR:", STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", STEP6_OUTPUT_DIR)
print("FEATURE_KEY:", FEATURE_KEY)
print("LABEL_FIELD:", LABEL_FIELD)
print("TRAIN_SCENES:", TRAIN_SCENES)
print("TEST_SCENES:", TEST_SCENES)

print("\nCurrent Step6 task:")
print("Decode clean LRAB implicit-transitive relation from hidden-state features.")
print("ALLOWED_EVIDENCE_TYPES:", sorted(ALLOWED_EVIDENCE_TYPES))
print("PROBE_RELATION_LABELS:", sorted(PROBE_RELATION_LABELS))
print("ALLOWED_LABELS:", sorted(ALLOWED_LABELS))
print("EXPECTED_RELATION_LABELS:", sorted(EXPECTED_RELATION_LABELS))
print("GROUP_BY_IMPLICIT_RULE:", GROUP_BY_IMPLICIT_RULE)

Synthetic LRAB Step6 config loaded.
STEP4_OUTPUT_BASENAME: pilot4_synthetic_disorder_lrab_2500
EXPERIMENT_NAME: pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe
MODEL_NAME: Qwen/Qwen2.5-0.5B
MODEL_TAG: Qwen2_5_0_5B
STEP6_INPUT_DIR: /content/pilot_4/data/pilot4_synthetic_disorder_lrab_2500_step6_input_hidden_states_qwen2_5_0_5b
STEP6_OUTPUT_DIR: /content/pilot_4/data/pilot4_synthetic_disorder_lrab_2500_step6_outputs_implicit_transitive_lrab_qwen2_5_0_5b
FEATURE_KEY: layer_diff_features
LABEL_FIELD: relation
TRAIN_SCENES: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4']
TEST_SCENES: ['FloorPlan5', 'FloorPlan6']

Current Step6 task:
Decode clean LRAB implicit-transitive relation from hidden-state features.
ALLOWED_EVIDENCE_TYPES: ['implicit_transitive']
PROBE_RELATION_LABELS: ['above', 'below', 'left_of', 'right_of']
ALLOWED_LABELS: ['above', 'below', 'left_of', 'right_of']
EXPECTED_RELATION_LABELS: ['above', 'below', 'left_of', 'right_of']
GROUP_BY_IMPLICIT_RULE: Tru

In [3]:
# ============================================================
# Helper functions for loading Step 5 records
# ============================================================

def discover_step5_pt_files(input_dir: str) -> List[str]:
    pt_paths = sorted(Path(input_dir).rglob("*.pt"))

    if not pt_paths:
        raise FileNotFoundError(f"No .pt files found in STEP6_INPUT_DIR: {input_dir}")

    return [str(p) for p in pt_paths]


def normalize_record(raw_record: Dict[str, Any], source_pt_file: str) -> Dict[str, Any]:
    """
    Normalize one Step 5 record into a predictable structure.

    Expected current Step5 fields:
      - layer_diff_features / layer_concat_features / ...
      - relation
      - scene
      - evidence_type = implicit_transitive
      - implicit_rule
      - anchor_uid
      - subject_alias / object_alias
    """

    rec = dict(raw_record)
    rec["_source_pt_file"] = source_pt_file

    # Some versions save metadata as rec["metadata"], while others flatten it.
    metadata = rec.get("metadata", {})

    if metadata is None:
        metadata = {}

    if not isinstance(metadata, dict):
        metadata = {}

    for k, v in metadata.items():
        rec.setdefault(k, v)

    # If nested evidence exists, expose useful fields.
    evidence = rec.get("evidence", {})
    if isinstance(evidence, dict):
        rec.setdefault("evidence_type", evidence.get("evidence_type"))
        rec.setdefault(
            "probe_direction_relative_to_text",
            evidence.get("probe_direction_relative_to_text"),
        )
        rec.setdefault("implicit_rule", evidence.get("implicit_rule"))
        rec.setdefault("anchor_uid", evidence.get("anchor_uid"))
        rec.setdefault("supporting_relations", evidence.get("supporting_relations"))
        rec.setdefault("num_supporting_paths", evidence.get("num_supporting_paths"))

    # If nested probe_pair exists, expose useful fields.
    probe_pair = rec.get("probe_pair", {})
    if isinstance(probe_pair, dict):
        rec.setdefault("probe_subject_uid", probe_pair.get("probe_subject_uid"))
        rec.setdefault("probe_object_uid", probe_pair.get("probe_object_uid"))
        rec.setdefault("probe_relation_label", probe_pair.get("probe_relation_label"))
        rec.setdefault("is_implicit_transitive", probe_pair.get("is_implicit_transitive"))
        rec.setdefault("is_explicit_in_text", probe_pair.get("is_explicit_in_text"))

    # Standardize relation label.
    if LABEL_FIELD not in rec:
        if "relation" in rec:
            rec[LABEL_FIELD] = rec["relation"]
        elif "probe_relation_label" in rec:
            rec[LABEL_FIELD] = rec["probe_relation_label"]
        elif isinstance(probe_pair, dict) and "probe_relation_label" in probe_pair:
            rec[LABEL_FIELD] = probe_pair["probe_relation_label"]
        else:
            raise KeyError(f"Could not find label field '{LABEL_FIELD}' in record.")

    # Standardize scene.
    if "scene" not in rec:
        if "source_step3_scene" in rec:
            rec["scene"] = rec["source_step3_scene"]
        else:
            raise KeyError("Could not find scene field in record.")

    # Standardize subject/object aliases if needed.
    if "subject_alias" not in rec:
        rec["subject_alias"] = (
            rec.get("probe_subject_uid")
            or rec.get("subject_uid")
            or rec.get("subject")
        )

    if "object_alias" not in rec:
        rec["object_alias"] = (
            rec.get("probe_object_uid")
            or rec.get("object_uid")
            or rec.get("object")
        )

    return rec


def extract_records_from_payload(payload: Any, source_pt_file: str) -> List[Dict[str, Any]]:
    """
    Supports several possible Step 5 payload formats:
      - list[record]
      - dict with key "records"
      - dict with key "data"
      - dict with key "examples"
    """

    if isinstance(payload, list):
        raw_records = payload
    elif isinstance(payload, dict):
        if "records" in payload:
            raw_records = payload["records"]
        elif "data" in payload:
            raw_records = payload["data"]
        elif "examples" in payload:
            raw_records = payload["examples"]
        else:
            raise KeyError(
                f"Unsupported .pt payload keys in {source_pt_file}: {list(payload.keys())}"
            )
    else:
        raise TypeError(f"Unsupported .pt payload type in {source_pt_file}: {type(payload)}")

    records = [
        normalize_record(r, source_pt_file=source_pt_file)
        for r in raw_records
    ]

    return records


def to_numpy_feature(x: Any) -> np.ndarray:
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().float().numpy()

    arr = np.asarray(x)

    if arr.dtype == np.dtype("O"):
        arr = arr.astype(np.float32)

    return arr.astype(np.float32, copy=False)


print("Helper functions loaded.")

Helper functions loaded.


In [4]:
# ============================================================
# Load Step 5 records, check Step5 label coverage,
# filter implicit-transitive samples,
# and build feature tensor
# ============================================================

pt_files = discover_step5_pt_files(STEP6_INPUT_DIR)

all_records_raw = []

for pt_path in pt_files:
    payload = torch.load(pt_path, map_location="cpu")
    records = extract_records_from_payload(payload, source_pt_file=pt_path)
    all_records_raw.extend(records)

if not all_records_raw:
    raise ValueError("No Step 5 records loaded.")

print("Number of raw Step 5 records loaded:", len(all_records_raw))
print("Number of source .pt files:", len(pt_files))

# ------------------------------------------------------------
# Step5 label coverage check before Step6 filtering
# ------------------------------------------------------------

raw_label_counts = Counter()
raw_evidence_counts = Counter()
raw_label_by_evidence_counts = defaultdict(Counter)

for r in all_records_raw:
    evidence_type = r.get("evidence_type")
    label = r.get(LABEL_FIELD)

    raw_evidence_counts[evidence_type] += 1

    if label is not None:
        raw_label_counts[label] += 1
        raw_label_by_evidence_counts[evidence_type][label] += 1

print("\n=== Step5 raw relation label counts ===")
for label, count in sorted(raw_label_counts.items()):
    print(f"{str(label):12s} {count}")

print("\n=== Step5 raw evidence type counts ===")
for evidence_type, count in sorted(raw_evidence_counts.items(), key=lambda x: str(x[0])):
    print(f"{str(evidence_type):25s} {count}")

print("\n=== Step5 relation label counts by evidence type ===")
for evidence_type, counter in sorted(raw_label_by_evidence_counts.items(), key=lambda x: str(x[0])):
    print(f"\n[{evidence_type}]")
    for label, count in sorted(counter.items()):
        print(f"{str(label):12s} {count}")

implicit_raw_label_counts = raw_label_by_evidence_counts.get("implicit_transitive", Counter())
implicit_raw_labels = set(implicit_raw_label_counts.keys())

missing_expected_labels = sorted(EXPECTED_RELATION_LABELS - implicit_raw_labels)
extra_implicit_labels = sorted(implicit_raw_labels - EXPECTED_RELATION_LABELS)

print("\n=== Step5 implicit_transitive label coverage check ===")
print("Expected labels:", sorted(EXPECTED_RELATION_LABELS))
print("Found implicit_transitive labels:", sorted(implicit_raw_labels))
print("Missing expected labels:", missing_expected_labels)
print("Extra implicit_transitive labels:", extra_implicit_labels)

if missing_expected_labels:
    print(
        "\nWarning: Some EXPECTED_RELATION_LABELS are missing from "
        "Step5 implicit_transitive records. "
        "Step6 will continue with the labels that are actually available."
    )

# ------------------------------------------------------------
# Filter to current ExpB SettingA1 target samples
# ------------------------------------------------------------

filtered_records = []

for r in all_records_raw:
    evidence_type = r.get("evidence_type")
    label = r.get(LABEL_FIELD)

    if evidence_type not in ALLOWED_EVIDENCE_TYPES:
        continue

    if ALLOWED_LABELS is not None and label not in ALLOWED_LABELS:
        continue

    filtered_records.append(r)

if not filtered_records:
    allowed_labels_display = (
        "ALL_LABELS_IN_STEP5"
        if ALLOWED_LABELS is None
        else sorted(ALLOWED_LABELS)
    )

    raise ValueError(
        "No records left after filtering. "
        f"ALLOWED_EVIDENCE_TYPES={ALLOWED_EVIDENCE_TYPES}, "
        f"ALLOWED_LABELS={allowed_labels_display}"
    )

all_records = filtered_records

print("\nNumber of records after filtering:", len(all_records))

# ------------------------------------------------------------
# Validate feature and label fields
# ------------------------------------------------------------

missing_feature = [i for i, r in enumerate(all_records) if FEATURE_KEY not in r]
missing_label = [i for i, r in enumerate(all_records) if LABEL_FIELD not in r]

if missing_feature:
    raise KeyError(f"{len(missing_feature)} records are missing feature key: {FEATURE_KEY}")

if missing_label:
    raise KeyError(f"{len(missing_label)} records are missing label field: {LABEL_FIELD}")

# ------------------------------------------------------------
# Stack features
# Expected shape per record: [num_layers, hidden_dim]
# ------------------------------------------------------------

features = []

for r in all_records:
    f = to_numpy_feature(r[FEATURE_KEY])

    if f.ndim != 2:
        raise ValueError(
            f"Expected feature shape [num_layers, dim], got {f.shape} "
            f"for sample_id={r.get('sample_id')}"
        )

    features.append(f)

X_all = np.stack(features, axis=0)  # [num_samples, num_layers, dim]
y_all = np.array([r[LABEL_FIELD] for r in all_records])
metadata = all_records

num_samples, num_layers, feature_dim = X_all.shape

print("\nX_all shape:", X_all.shape)
print("num_samples:", num_samples)
print("num_layers:", num_layers)
print("feature_dim:", feature_dim)

print("\nLabel counts after Step6 filtering:")
print(dict(Counter(y_all)))

print("\nScene counts:")
print(dict(Counter(r["scene"] for r in metadata)))

print("\nEvidence type counts:")
print(dict(Counter(r.get("evidence_type", "unknown") for r in metadata)))

print("\nImplicit rule counts:")
print(dict(Counter(r.get("implicit_rule", "unknown") for r in metadata)))

Number of raw Step 5 records loaded: 10000
Number of source .pt files: 1

=== Step5 raw relation label counts ===
above        2500
below        2500
left_of      2500
right_of     2500

=== Step5 raw evidence type counts ===
implicit_transitive       10000

=== Step5 relation label counts by evidence type ===

[implicit_transitive]
above        2500
below        2500
left_of      2500
right_of     2500

=== Step5 implicit_transitive label coverage check ===
Expected labels: ['above', 'below', 'left_of', 'right_of']
Found implicit_transitive labels: ['above', 'below', 'left_of', 'right_of']
Missing expected labels: []
Extra implicit_transitive labels: []

Number of records after filtering: 10000

X_all shape: (10000, 25, 896)
num_samples: 10000
num_layers: 25
feature_dim: 896

Label counts after Step6 filtering:
{np.str_('below'): 2500, np.str_('right_of'): 2500, np.str_('left_of'): 2500, np.str_('above'): 2500}

Scene counts:
{'FloorPlan1': 1999, 'FloorPlan6': 999, 'FloorPlan4': 2000,

In [5]:
# ============================================================
# Scene split and common-label filtering
#
# Goal:
#   train on TRAIN_SCENES
#   test on TEST_SCENES
#
# Labels:
#   all implicit-transitive relation labels available in both
#   train and test scenes.
#
# Important:
#   Labels that appear only in train or only in test are removed
#   by the common-label filter below.
# ============================================================

all_scenes = sorted(set(r["scene"] for r in metadata))

if REQUIRE_EXPLICIT_SCENE_SPLIT:
    missing_train = sorted(set(TRAIN_SCENES) - set(all_scenes))
    missing_test = sorted(set(TEST_SCENES) - set(all_scenes))

    if missing_train or missing_test:
        raise ValueError(
            f"Scene split references missing scenes. "
            f"missing_train={missing_train}, missing_test={missing_test}, "
            f"available_scenes={all_scenes}"
        )

train_idx = np.array(
    [i for i, r in enumerate(metadata) if r["scene"] in TRAIN_SCENES],
    dtype=np.int64,
)

test_idx = np.array(
    [i for i, r in enumerate(metadata) if r["scene"] in TEST_SCENES],
    dtype=np.int64,
)

if len(train_idx) == 0:
    raise ValueError("No training examples selected.")

if len(test_idx) == 0:
    raise ValueError("No test examples selected.")

train_labels_before = set(y_all[train_idx])
test_labels_before = set(y_all[test_idx])
common_labels = sorted(train_labels_before & test_labels_before)

if len(common_labels) < 2:
    raise ValueError(
        f"Need at least two common labels for classification. "
        f"train_labels={sorted(train_labels_before)}, "
        f"test_labels={sorted(test_labels_before)}, "
        f"common_labels={common_labels}"
    )

removed_train_only_labels = sorted(train_labels_before - test_labels_before)
removed_test_only_labels = sorted(test_labels_before - train_labels_before)

train_idx = np.array(
    [idx for idx in train_idx if y_all[idx] in common_labels],
    dtype=np.int64,
)

test_idx = np.array(
    [idx for idx in test_idx if y_all[idx] in common_labels],
    dtype=np.int64,
)

label_order = common_labels

scene_split_info = {
    "train_scenes": TRAIN_SCENES,
    "test_scenes": TEST_SCENES,
    "available_scenes": all_scenes,

    "num_train_before_common_label_filter": int(sum(r["scene"] in TRAIN_SCENES for r in metadata)),
    "num_test_before_common_label_filter": int(sum(r["scene"] in TEST_SCENES for r in metadata)),

    "train_labels_before": sorted(train_labels_before),
    "test_labels_before": sorted(test_labels_before),
    "common_labels": common_labels,
    "removed_train_only_labels": removed_train_only_labels,
    "removed_test_only_labels": removed_test_only_labels,

    "num_train_final": int(len(train_idx)),
    "num_test_final": int(len(test_idx)),

    "train_label_counts_final": dict(Counter(y_all[train_idx])),
    "test_label_counts_final": dict(Counter(y_all[test_idx])),

    "train_scene_counts_final": dict(Counter(metadata[int(idx)]["scene"] for idx in train_idx)),
    "test_scene_counts_final": dict(Counter(metadata[int(idx)]["scene"] for idx in test_idx)),

    "train_evidence_type_counts_final": dict(
        Counter(metadata[int(idx)].get("evidence_type", "unknown") for idx in train_idx)
    ),
    "test_evidence_type_counts_final": dict(
        Counter(metadata[int(idx)].get("evidence_type", "unknown") for idx in test_idx)
    ),

    "train_implicit_rule_counts_final": dict(
        Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in train_idx)
    ),
    "test_implicit_rule_counts_final": dict(
        Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in test_idx)
    ),
}

print("Scene split info:")
print(json.dumps(scene_split_info, indent=2, ensure_ascii=False))

Scene split info:
{
  "train_scenes": [
    "FloorPlan1",
    "FloorPlan2",
    "FloorPlan3",
    "FloorPlan4"
  ],
  "test_scenes": [
    "FloorPlan5",
    "FloorPlan6"
  ],
  "available_scenes": [
    "FloorPlan1",
    "FloorPlan2",
    "FloorPlan3",
    "FloorPlan4",
    "FloorPlan5",
    "FloorPlan6"
  ],
  "num_train_before_common_label_filter": 8000,
  "num_test_before_common_label_filter": 2000,
  "train_labels_before": [
    "above",
    "below",
    "left_of",
    "right_of"
  ],
  "test_labels_before": [
    "above",
    "below",
    "left_of",
    "right_of"
  ],
  "common_labels": [
    "above",
    "below",
    "left_of",
    "right_of"
  ],
  "removed_train_only_labels": [],
  "removed_test_only_labels": [],
  "num_train_final": 8000,
  "num_test_final": 2000,
  "train_label_counts_final": {
    "below": 2000,
    "left_of": 2000,
    "above": 2000,
    "right_of": 2000
  },
  "test_label_counts_final": {
    "right_of": 500,
    "left_of": 500,
    "below": 500,
    "abo

In [6]:
# ============================================================
# Optional sanity check: train/test object UID split
# ============================================================

def collect_object_uids(records, indices):
    uids = set()

    for idx in indices:
        r = records[int(idx)]

        for key in [
            "probe_subject_uid",
            "probe_object_uid",
            "anchor_uid",
            "subject_alias",
            "object_alias",
        ]:
            v = r.get(key)
            if isinstance(v, str):
                uids.add(v)

        for sr in r.get("supporting_relations", []) or []:
            if isinstance(sr, dict):
                for key in ["subject_uid", "object_uid"]:
                    v = sr.get(key)
                    if isinstance(v, str):
                        uids.add(v)

    return uids


train_uids = collect_object_uids(metadata, train_idx)
test_uids = collect_object_uids(metadata, test_idx)
overlap = train_uids & test_uids

print("train object uid count:", len(train_uids))
print("test object uid count:", len(test_uids))
print("train/test object uid overlap:", len(overlap))

if overlap:
    print("Example overlaps:", list(sorted(overlap))[:20])
    raise ValueError("Train/test object UID overlap detected.")

train object uid count: 1800
test object uid count: 900
train/test object uid overlap: 0


In [7]:
# ============================================================
# No direct/inverse direction filtering for current Step6
#
# Current data:
#   evidence_type = implicit_transitive
#   probe_direction_relative_to_text = implicit
#
# Therefore, old direct/inverse filtering is disabled.
# ============================================================

direction_filter_info = {
    "enabled": False,
    "reason": (
        "Current Step6 uses Step4T implicit-transitive samples. "
        "There is no direct/inverse train-test direction protocol here."
    ),
}

print("Direction filtering disabled for current implicit-transitive Step6.")
print(json.dumps(direction_filter_info, indent=2, ensure_ascii=False))

Direction filtering disabled for current implicit-transitive Step6.
{
  "enabled": false,
  "reason": "Current Step6 uses Step4T implicit-transitive samples. There is no direct/inverse train-test direction protocol here."
}


In [8]:
# ============================================================
# Final train/test split summary
# ============================================================

final_split_summary = {
    "num_samples_loaded_after_filtering": int(num_samples),
    "num_train": int(len(train_idx)),
    "num_test": int(len(test_idx)),
    "label_order": label_order,

    "train_label_counts": dict(Counter(y_all[train_idx])),
    "test_label_counts": dict(Counter(y_all[test_idx])),

    "train_scene_counts": dict(Counter(metadata[int(idx)]["scene"] for idx in train_idx)),
    "test_scene_counts": dict(Counter(metadata[int(idx)]["scene"] for idx in test_idx)),

    "train_implicit_rule_counts": dict(
        Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in train_idx)
    ),
    "test_implicit_rule_counts": dict(
        Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in test_idx)
    ),

    "feature_key": FEATURE_KEY,
    "label_field": LABEL_FIELD,
}

print("Final train/test split summary:")
print(json.dumps(final_split_summary, indent=2, ensure_ascii=False))

Final train/test split summary:
{
  "num_samples_loaded_after_filtering": 10000,
  "num_train": 8000,
  "num_test": 2000,
  "label_order": [
    "above",
    "below",
    "left_of",
    "right_of"
  ],
  "train_label_counts": {
    "below": 2000,
    "left_of": 2000,
    "above": 2000,
    "right_of": 2000
  },
  "test_label_counts": {
    "right_of": 500,
    "left_of": 500,
    "below": 500,
    "above": 500
  },
  "train_scene_counts": {
    "FloorPlan1": 1999,
    "FloorPlan4": 2000,
    "FloorPlan3": 2000,
    "FloorPlan2": 2001
  },
  "test_scene_counts": {
    "FloorPlan6": 999,
    "FloorPlan5": 1001
  },
  "train_implicit_rule_counts": {
    "anchor_between_reversed_surface_form": 2668,
    "shared_anchor_opposite_sides": 2667,
    "chain_same_direction": 2665
  },
  "test_implicit_rule_counts": {
    "chain_same_direction": 669,
    "shared_anchor_opposite_sides": 666,
    "anchor_between_reversed_surface_form": 665
  },
  "feature_key": "layer_diff_features",
  "label_field"

In [9]:
# ============================================================
# Build test subsets
#
# Current subsets:
#   - overall test set
#   - optional subsets by implicit_rule
# ============================================================

test_subset_indices = {
    "test_overall": test_idx,
}

if GROUP_BY_IMPLICIT_RULE:
    test_rules = sorted(
        set(
            metadata[int(idx)].get("implicit_rule", "unknown")
            for idx in test_idx
        )
    )

    for rule in test_rules:
        subset_key = f"test_rule_{rule}"
        rule_idx = np.array(
            [
                idx for idx in test_idx
                if metadata[int(idx)].get("implicit_rule", "unknown") == rule
            ],
            dtype=np.int64,
        )
        test_subset_indices[subset_key] = rule_idx

test_subset_summary = {}

for subset_key, idxs in test_subset_indices.items():
    test_subset_summary[subset_key] = {
        "num_examples": int(len(idxs)),
        "label_counts": dict(Counter(y_all[idxs])) if len(idxs) else {},
        "evidence_type_counts": dict(
            Counter(metadata[int(idx)].get("evidence_type", "unknown") for idx in idxs)
        ) if len(idxs) else {},
        "implicit_rule_counts": dict(
            Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in idxs)
        ) if len(idxs) else {},
        "scene_counts": dict(
            Counter(metadata[int(idx)].get("scene", "unknown") for idx in idxs)
        ) if len(idxs) else {},
    }

print("Test subset summary:")
print(json.dumps(test_subset_summary, indent=2, ensure_ascii=False))

Test subset summary:
{
  "test_overall": {
    "num_examples": 2000,
    "label_counts": {
      "right_of": 500,
      "left_of": 500,
      "below": 500,
      "above": 500
    },
    "evidence_type_counts": {
      "implicit_transitive": 2000
    },
    "implicit_rule_counts": {
      "chain_same_direction": 669,
      "shared_anchor_opposite_sides": 666,
      "anchor_between_reversed_surface_form": 665
    },
    "scene_counts": {
      "FloorPlan6": 999,
      "FloorPlan5": 1001
    }
  },
  "test_rule_anchor_between_reversed_surface_form": {
    "num_examples": 665,
    "label_counts": {
      "below": 166,
      "left_of": 166,
      "above": 166,
      "right_of": 167
    },
    "evidence_type_counts": {
      "implicit_transitive": 665
    },
    "implicit_rule_counts": {
      "anchor_between_reversed_surface_form": 665
    },
    "scene_counts": {
      "FloorPlan5": 333,
      "FloorPlan6": 332
    }
  },
  "test_rule_chain_same_direction": {
    "num_examples": 669,
    "

In [10]:
# ============================================================
# Layer-wise training and evaluation
# ============================================================

def evaluate_classifier(
    clf: LogisticRegression,
    X_eval: np.ndarray,
    y_eval: np.ndarray,
    labels: List[str],
) -> Dict[str, Any]:
    if len(y_eval) == 0:
        return {
            "num_examples": 0,
            "accuracy": None,
            "macro_f1": None,
            "label_order": labels,
            "classification_report": {},
            "confusion_matrix": [],
        }

    y_pred = clf.predict(X_eval)

    acc = accuracy_score(y_eval, y_pred)
    macro_f1 = f1_score(
        y_eval,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0,
    )

    report = classification_report(
        y_eval,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_eval,
        y_pred,
        labels=labels,
    )

    return {
        "num_examples": int(len(y_eval)),
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "label_order": labels,
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
    }


results_by_layer = []

for layer_id in range(num_layers):
    if PRINT_LAYER_PROGRESS:
        print(f"Training layer {layer_id}/{num_layers - 1}")

    X_train_layer = X_all[train_idx, layer_id, :]
    y_train_layer = y_all[train_idx]

    clf = LogisticRegression(
        max_iter=LOGREG_MAX_ITER,
        C=LOGREG_C,
        multi_class="auto",
        solver="lbfgs",
        random_state=RANDOM_SEED,
    )

    clf.fit(X_train_layer, y_train_layer)

    layer_result = {
        "layer": int(layer_id),
        "train": evaluate_classifier(
            clf,
            X_train_layer,
            y_train_layer,
            labels=label_order,
        ),
    }

    for subset_key, idxs in test_subset_indices.items():
        X_eval = X_all[idxs, layer_id, :]
        y_eval = y_all[idxs]

        layer_result[subset_key] = evaluate_classifier(
            clf,
            X_eval,
            y_eval,
            labels=label_order,
        )

    results_by_layer.append(layer_result)

print("Layer-wise training and evaluation complete.")

Training layer 0/24


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training layer 1/24


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training layer 2/24


...

Training layer 23/24


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training layer 24/24


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer-wise training and evaluation complete.


In [18]:
# ============================================================
# Summarize top layers
# and export per-layer per-label metrics
# ============================================================

summary_rows = []
per_label_metric_rows = []

test_subset_keys = list(test_subset_indices.keys())


def build_per_label_rows_from_eval_result(
    *,
    layer: int,
    subset_name: str,
    eval_result: Dict[str, Any],
) -> List[Dict[str, Any]]:
    """
    Convert one evaluate_classifier() result into per-label rows.

    This uses:
      - eval_result["classification_report"]
      - eval_result["confusion_matrix"]
      - eval_result["label_order"]

    Metrics exported per label:
      - precision
      - recall
      - f1
      - support
      - label_accuracy_true_class = TP / (TP + FN)
      - one_vs_rest_accuracy = (TP + TN) / total
      - tp, fp, fn, tn

    Note:
      In multi-class classification, "per-label accuracy" is ambiguous.
      Therefore we export both:
        1. label_accuracy_true_class:
           accuracy among examples whose true label is this label.
           This is numerically the same as recall.

        2. one_vs_rest_accuracy:
           accuracy after treating this label as positive and all other labels
           as negative.
    """

    if eval_result["num_examples"] == 0:
        return []

    labels = list(eval_result["label_order"])
    report = eval_result["classification_report"]
    cm = np.array(eval_result["confusion_matrix"], dtype=int)

    if cm.size == 0:
        return []

    total = int(cm.sum())
    rows = []

    for i, label in enumerate(labels):
        tp = int(cm[i, i])
        fn = int(cm[i, :].sum() - tp)
        fp = int(cm[:, i].sum() - tp)
        tn = int(total - tp - fn - fp)

        support = int(tp + fn)

        if support > 0:
            label_accuracy_true_class = tp / support
        else:
            label_accuracy_true_class = 0.0

        if total > 0:
            one_vs_rest_accuracy = (tp + tn) / total
        else:
            one_vs_rest_accuracy = 0.0

        label_report = report.get(label, {})

        rows.append({
            "layer": int(layer),
            "subset": subset_name,
            "label": label,

            # Two explicit definitions of per-label accuracy.
            "label_accuracy_true_class": float(label_accuracy_true_class),
            "one_vs_rest_accuracy": float(one_vs_rest_accuracy),

            # Standard classification-report metrics.
            "precision": float(label_report.get("precision", 0.0)),
            "recall": float(label_report.get("recall", 0.0)),
            "f1": float(label_report.get("f1-score", 0.0)),
            "support": int(label_report.get("support", support)),

            # Confusion-matrix components.
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            "total": total,
        })

    return rows


for r in results_by_layer:
    row = {
        "layer": r["layer"],
        "train_accuracy": r["train"]["accuracy"],
        "train_macro_f1": r["train"]["macro_f1"],
    }

    # Per-label metrics for train set.
    per_label_metric_rows.extend(
        build_per_label_rows_from_eval_result(
            layer=r["layer"],
            subset_name="train",
            eval_result=r["train"],
        )
    )

    # Per-layer overall metrics and per-label metrics for each test subset.
    for subset_key in test_subset_keys:
        row[f"{subset_key}_accuracy"] = r[subset_key]["accuracy"]
        row[f"{subset_key}_macro_f1"] = r[subset_key]["macro_f1"]
        row[f"{subset_key}_num_examples"] = r[subset_key]["num_examples"]

        per_label_metric_rows.extend(
            build_per_label_rows_from_eval_result(
                layer=r["layer"],
                subset_name=subset_key,
                eval_result=r[subset_key],
            )
        )

    summary_rows.append(row)


# ------------------------------------------------------------
# 1. Original layer-level summary CSV
# ------------------------------------------------------------

summary_df = pd.DataFrame(summary_rows)

summary_csv_path = os.path.join(
    STEP6_OUTPUT_DIR,
    f"step6_{EXPERIMENT_NAME}_{MODEL_TAG}_{FEATURE_KEY}_layer_summary.csv",
)

summary_df.to_csv(summary_csv_path, index=False)

print("Saved layer summary CSV:")
print(summary_csv_path)

display(summary_df.head())


# ------------------------------------------------------------
# 2. New long-format per-layer per-label metrics CSV
# ------------------------------------------------------------

per_label_metrics_df = pd.DataFrame(per_label_metric_rows)

per_label_metrics_csv_path = os.path.join(
    STEP6_OUTPUT_DIR,
    f"step6_{EXPERIMENT_NAME}_{MODEL_TAG}_{FEATURE_KEY}_per_layer_per_label_metrics.csv",
)

per_label_metrics_df.to_csv(
    per_label_metrics_csv_path,
    index=False,
    encoding="utf-8",
)

print("\nSaved per-layer per-label metrics CSV:")
print(per_label_metrics_csv_path)

display(per_label_metrics_df.head(20))


# ------------------------------------------------------------
# 3. Optional wide-format per-layer per-label metrics CSV
# ------------------------------------------------------------

if not per_label_metrics_df.empty:
    wide_per_label_metrics_df = per_label_metrics_df.pivot_table(
        index=["layer", "subset"],
        columns="label",
        values=[
            "label_accuracy_true_class",
            "one_vs_rest_accuracy",
            "precision",
            "recall",
            "f1",
            "support",
        ],
        aggfunc="first",
    )

    wide_per_label_metrics_df.columns = [
        f"{metric}_{label}"
        for metric, label in wide_per_label_metrics_df.columns
    ]

    wide_per_label_metrics_df = wide_per_label_metrics_df.reset_index()

    wide_per_label_metrics_csv_path = os.path.join(
        STEP6_OUTPUT_DIR,
        f"step6_{EXPERIMENT_NAME}_{MODEL_TAG}_{FEATURE_KEY}_per_layer_per_label_metrics_wide.csv",
    )

    wide_per_label_metrics_df.to_csv(
        wide_per_label_metrics_csv_path,
        index=False,
        encoding="utf-8",
    )

    print("\nSaved wide per-layer per-label metrics CSV:")
    print(wide_per_label_metrics_csv_path)

    display(wide_per_label_metrics_df.head())
else:
    wide_per_label_metrics_df = pd.DataFrame()
    wide_per_label_metrics_csv_path = None
    print("\nWarning: per_label_metrics_df is empty. Wide CSV was not created.")


# ------------------------------------------------------------
# 4. Top-layer display, keeping your original behavior
# ------------------------------------------------------------

if PRINT_TOP_LAYERS:
    ranking_metrics = [
        "test_overall_accuracy",
        "test_overall_macro_f1",
    ]

    # Also print rule-specific metrics if available.
    for subset_key in test_subset_keys:
        if subset_key != "test_overall":
            ranking_metrics.append(f"{subset_key}_accuracy")

    for metric in ranking_metrics:
        if metric not in summary_df.columns:
            continue

        print("\n" + "=" * 80)
        print("Top layers by:", metric)
        print("=" * 80)

        cols = [
            "layer",
            metric,
            "test_overall_accuracy",
            "test_overall_macro_f1",
        ]

        # Add available subset accuracy columns.
        for subset_key in test_subset_keys:
            acc_col = f"{subset_key}_accuracy"
            if acc_col not in cols and acc_col in summary_df.columns:
                cols.append(acc_col)

        display(
            summary_df.sort_values(metric, ascending=False)
            .loc[:, cols]
            .head(NUM_TOP_LAYERS_TO_PRINT)
        )

Saved layer summary CSV:
/content/pilot_4/data/pilot4_synthetic_disorder_lrab_2500_step6_outputs_implicit_transitive_lrab_qwen2_5_0_5b/step6_pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe_Qwen2_5_0_5B_layer_diff_features_layer_summary.csv


,layer,train_accuracy,train_macro_f1,test_overall_accuracy,test_overall_macro_f1,test_overall_num_examples,test_rule_anchor_between_reversed_surface_form_accuracy,test_rule_anchor_between_reversed_surface_form_macro_f1,test_rule_anchor_between_reversed_surface_form_num_examples,test_rule_chain_same_direction_accuracy,test_rule_chain_same_direction_macro_f1,test_rule_chain_same_direction_num_examples,test_rule_shared_anchor_opposite_sides_accuracy,test_rule_shared_anchor_opposite_sides_macro_f1,test_rule_shared_anchor_opposite_sides_num_examples
0,0,0.250000,0.100000,0.2500,0.100000,2000,0.249624,0.099880,665,0.251121,0.100358,669,0.249249,0.099760,666
1,1,0.548125,0.546157,0.4740,0.471371,2000,0.494737,0.492155,665,0.442451,0.440743,669,0.484985,0.481446,666
2,2,0.793000,0.793391,0.7125,0.712803,2000,0.724812,0.725266,665,0.693572,0.693601,669,0.719219,0.718850,666
3,3,0.828250,0.828349,0.7440,0.744202,2000,0.763910,0.763429,665,0.715994,0.716225,669,0.752252,0.752175,666
4,4,0.817500,0.817529,0.6470,0.647025,2000,0.678195,0.678019,665,0.603886,0.603835,669,0.659159,0.658473,666



Saved per-layer per-label metrics CSV:
/content/pilot_4/data/pilot4_synthetic_disorder_lrab_2500_step6_outputs_implicit_transitive_lrab_qwen2_5_0_5b/step6_pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe_Qwen2_5_0_5B_layer_diff_features_per_layer_per_label_metrics.csv


,layer,subset,label,label_accuracy_true_class,one_vs_rest_accuracy,precision,recall,f1,support,tp,fp,fn,tn,total
0,0,train,above,1.0,0.250000,0.250000,1.0,0.400000,2000,2000,6000,0,0,8000
1,0,train,below,0.0,0.750000,0.000000,0.0,0.000000,2000,0,0,2000,6000,8000
2,0,train,left_of,0.0,0.750000,0.000000,0.0,0.000000,2000,0,0,2000,6000,8000
3,0,train,right_of,0.0,0.750000,0.000000,0.0,0.000000,2000,0,0,2000,6000,8000
4,0,test_overall,above,1.0,0.250000,0.250000,1.0,0.400000,500,500,1500,0,0,2000
5,0,test_overall,below,0.0,0.750000,0.000000,0.0,0.000000,500,0,0,500,1500,2000
6,0,test_overall,left_of,0.0,0.750000,0.000000,0.0,0.000000,500,0,0,500,1500,2000
7,0,test_overall,right_of,0.0,0.750000,0.000000,0.0,0.000000,500,0,0,500,1500,2000
8,0,test_rule_anchor_between_reversed_surface_form,above,1.0,0.249624,0.249624,1.0,0.399519,166,166,499,0,0,665
9,0,test_rule_anchor_between_reversed_surface_form,below,0.0,0.750376,0.000000,0.0,0.000000,166,0,0,166,499,665



Saved wide per-layer per-label metrics CSV:
/content/pilot_4/data/pilot4_synthetic_disorder_lrab_2500_step6_outputs_implicit_transitive_lrab_qwen2_5_0_5b/step6_pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe_Qwen2_5_0_5B_layer_diff_features_per_layer_per_label_metrics_wide.csv


...

,layer,test_rule_anchor_between_reversed_surface_form_accuracy,test_overall_accuracy,test_overall_macro_f1,test_rule_chain_same_direction_accuracy,test_rule_shared_anchor_opposite_sides_accuracy
3,3,0.763910,0.7440,0.744202,0.715994,0.752252
2,2,0.724812,0.7125,0.712803,0.693572,0.719219
9,9,0.712782,0.6865,0.688202,0.675635,0.671171
10,10,0.685714,0.6540,0.654972,0.624813,0.651652
4,4,0.678195,0.6470,0.647025,0.603886,0.659159
11,11,0.645113,0.6235,0.624943,0.612855,0.612613
5,5,0.637594,0.6250,0.625092,0.618834,0.618619
8,8,0.615038,0.6010,0.601112,0.591928,0.596096
12,12,0.580451,0.5505,0.550849,0.527653,0.543544
19,19,0.575940,0.5630,0.562887,0.559043,0.554054



Top layers by: test_rule_chain_same_direction_accuracy


,layer,test_rule_chain_same_direction_accuracy,test_overall_accuracy,test_overall_macro_f1,test_rule_anchor_between_reversed_surface_form_accuracy,test_rule_shared_anchor_opposite_sides_accuracy
3,3,0.715994,0.7440,0.744202,0.763910,0.752252
2,2,0.693572,0.7125,0.712803,0.724812,0.719219
9,9,0.675635,0.6865,0.688202,0.712782,0.671171
10,10,0.624813,0.6540,0.654972,0.685714,0.651652
5,5,0.618834,0.6250,0.625092,0.637594,0.618619
11,11,0.612855,0.6235,0.624943,0.645113,0.612613
4,4,0.603886,0.6470,0.647025,0.678195,0.659159
8,8,0.591928,0.6010,0.601112,0.615038,0.596096
19,19,0.559043,0.5630,0.562887,0.575940,0.554054
6,6,0.545590,0.5575,0.557800,0.553383,0.573574



Top layers by: test_rule_shared_anchor_opposite_sides_accuracy


,layer,test_rule_shared_anchor_opposite_sides_accuracy,test_overall_accuracy,test_overall_macro_f1,test_rule_anchor_between_reversed_surface_form_accuracy,test_rule_chain_same_direction_accuracy
3,3,0.752252,0.7440,0.744202,0.763910,0.715994
2,2,0.719219,0.7125,0.712803,0.724812,0.693572
9,9,0.671171,0.6865,0.688202,0.712782,0.675635
4,4,0.659159,0.6470,0.647025,0.678195,0.603886
10,10,0.651652,0.6540,0.654972,0.685714,0.624813
5,5,0.618619,0.6250,0.625092,0.637594,0.618834
11,11,0.612613,0.6235,0.624943,0.645113,0.612855
8,8,0.596096,0.6010,0.601112,0.615038,0.591928
6,6,0.573574,0.5575,0.557800,0.553383,0.545590
19,19,0.554054,0.5630,0.562887,0.575940,0.559043


In [19]:
# ============================================================
# Save full Step 6 result JSON
# ============================================================

allowed_labels_for_json = sorted(ALLOWED_LABELS)

output_payload = {
    "experiment_name": EXPERIMENT_NAME,
    "description": (
        "Synthetic disorder clean LRAB implicit-transitive relation decoding. "
        "A linear probe is trained on Step5 hidden-state features extracted from "
        "the clean LRAB synthetic disorder dataset. Step4 restricts implicit "
        "target relations and supporting chains to {left_of, right_of, above, below}. "
        "Other relation labels may appear only as distractor relation sentences "
        "inside the input text. This Step6 experiment probes only the four LRAB "
        "implicit target labels. The main feature is "
        "hidden(probe_subject) - hidden(probe_object)."
    ),

    "model_name": MODEL_NAME,
    "model_tag": MODEL_TAG,
    "feature_key": FEATURE_KEY,
    "label_field": LABEL_FIELD,

    "probe_target": {
        "task": "implicit_transitive_lrab_relation_label_decoding",
        "input_feature": FEATURE_KEY,
        "label": LABEL_FIELD,

        # Decisive probe label definition.
        # Only these labels are used as y labels in linear probing.
        "probe_relation_labels": sorted(PROBE_RELATION_LABELS),
        "allowed_labels": allowed_labels_for_json,
        "effective_label_order": label_order,

        "allowed_evidence_types": sorted(ALLOWED_EVIDENCE_TYPES),
        "expected_relation_labels_for_coverage_check": sorted(EXPECTED_RELATION_LABELS),
        "feature_interpretation": "hidden(probe_subject) - hidden(probe_object)",

        "input_dataset_note": (
            "The input Step5 .pt file is generated from the clean LRAB Step4 dataset. "
            "Step4 restricts implicit targets and supporting chains to "
            "{left_of, right_of, above, below}. Labels such as in, contains, on, "
            "supports, and near may appear as relation-based distractor sentences "
            "in the paragraph text, but they are not probe labels and are not used "
            "as supporting-chain labels. Step6 filters records by "
            "evidence_type=implicit_transitive and by the four LRAB probe labels."
        ),
    },

    "num_samples_loaded_after_filtering": int(num_samples),
    "num_layers": int(num_layers),
    "feature_dim": int(feature_dim),
    "label_order": label_order,

    "scene_split_info": scene_split_info,
    "direction_filter_info": direction_filter_info,
    "final_split_summary": final_split_summary,
    "test_subset_summary": test_subset_summary,

    "step6_config": {
        "step4_output_basename": STEP4_OUTPUT_BASENAME,
        "synthetic_experiment_name": SYNTHETIC_EXPERIMENT_NAME,
        "experiment_name": EXPERIMENT_NAME,
        "model_name": MODEL_NAME,
        "model_tag": MODEL_TAG,
        "feature_key": FEATURE_KEY,
        "label_field": LABEL_FIELD,
        "train_scenes": TRAIN_SCENES,
        "test_scenes": TEST_SCENES,

        "probe_relation_labels": sorted(PROBE_RELATION_LABELS),
        "allowed_labels": allowed_labels_for_json,
        "effective_label_order": label_order,

        "allowed_evidence_types": sorted(ALLOWED_EVIDENCE_TYPES),
        "expected_relation_labels_for_coverage_check": sorted(EXPECTED_RELATION_LABELS),
        "logreg_max_iter": LOGREG_MAX_ITER,
        "logreg_c": LOGREG_C,
        "random_seed": RANDOM_SEED,
        "group_by_implicit_rule": GROUP_BY_IMPLICIT_RULE,
    },

    "source_pt_files": [str(p) for p in pt_files],
    "results_by_layer": results_by_layer,
    "per_layer_per_label_metrics": per_label_metric_rows,
    "per_layer_per_label_metrics_csv": per_label_metrics_csv_path,
    "per_layer_per_label_metrics_wide_csv": wide_per_label_metrics_csv_path,
    }

output_json_path = os.path.join(
    STEP6_OUTPUT_DIR,
    f"step6_{EXPERIMENT_NAME}_{MODEL_TAG}_{FEATURE_KEY}.json",
)

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(output_payload, f, indent=2, ensure_ascii=False)

print("Saved full Step 6 result JSON:")
print(output_json_path)

Saved full Step 6 result JSON:
/content/pilot_4/data/pilot4_synthetic_disorder_lrab_2500_step6_outputs_implicit_transitive_lrab_qwen2_5_0_5b/step6_pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe_Qwen2_5_0_5B_layer_diff_features.json


In [20]:
# ============================================================
# Zip Step 6 outputs
# ============================================================

zip_base = f"/content/{EXPERIMENT_NAME}_step6_{MODEL_TAG}_{FEATURE_KEY}"

zip_path = shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=STEP6_OUTPUT_DIR,
)

print("Created Step 6 output zip:")
print(zip_path)

print("\nSTEP6_OUTPUT_DIR:")
print(STEP6_OUTPUT_DIR)

print("\nFiles included:")
for p in sorted(Path(STEP6_OUTPUT_DIR).rglob("*")):
    if p.is_file():
        print("-", p.relative_to(STEP6_OUTPUT_DIR), "|", round(p.stat().st_size / 1024 / 1024, 2), "MB")

Created Step 6 output zip:
/content/pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe_step6_Qwen2_5_0_5B_layer_diff_features.zip

STEP6_OUTPUT_DIR:
/content/pilot_4/data/pilot4_synthetic_disorder_lrab_2500_step6_outputs_implicit_transitive_lrab_qwen2_5_0_5b

Files included:
- pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe_Qwen2_5_0_5B_step6_outputs.zip | 0.03 MB
- step6_pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe_Qwen2_5_0_5B_layer_diff_features.json | 0.43 MB
- step6_pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe_Qwen2_5_0_5B_layer_diff_features_layer_summary.csv | 0.0 MB
- step6_pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe_Qwen2_5_0_5B_layer_diff_features_per_layer_per_label_metrics.csv | 0.07 MB
- step6_pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe_Qwen2_5_0_5B_layer_diff_features_per_layer_per_label_metrics_wide.csv | 0.04 MB


In [21]:
# ============================================================
# Create Step 6 output zip and save it to Google Drive
# ============================================================

from pathlib import Path
import shutil
import zipfile

# ------------------------------------------------------------
# 1. Create local Step6 zip from all files in STEP6_OUTPUT_DIR
# ------------------------------------------------------------

step6_output_dir = Path(STEP6_OUTPUT_DIR)

if not step6_output_dir.exists():
    raise FileNotFoundError(
        "STEP6_OUTPUT_DIR does not exist.\n"
        f"STEP6_OUTPUT_DIR:\n{step6_output_dir}"
    )

files_to_zip = [
    p for p in sorted(step6_output_dir.rglob("*"))
    if p.is_file() and p.suffix != ".zip"
]

if not files_to_zip:
    raise FileNotFoundError(
        "No files found to zip in STEP6_OUTPUT_DIR.\n"
        "Please make sure the previous Step6 result-saving cell has run.\n"
        f"STEP6_OUTPUT_DIR:\n{step6_output_dir}"
    )

zip_path = step6_output_dir / f"{EXPERIMENT_NAME}_{MODEL_TAG}_step6_outputs.zip"

if zip_path.exists():
    zip_path.unlink()

print("Creating local Step6 zip:")
print(zip_path)

with zipfile.ZipFile(str(zip_path), "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in files_to_zip:
        arcname = p.relative_to(step6_output_dir)
        zf.write(str(p), arcname=str(arcname))

print("Created local zip:")
print(zip_path)
print("Zip size MB:", round(zip_path.stat().st_size / 1024 / 1024, 2))

# ------------------------------------------------------------
# 2. Save Step6 zip to Google Drive
# ------------------------------------------------------------

DRIVE_STEP6_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/"
    "expB_settingA1_6fps/setting_synthetic/"
    "data_synthetic_disorder_lrab_2500/"
    "step6_outputs_lrab_2500"
)

DRIVE_STEP6_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

drive_zip_path = DRIVE_STEP6_OUTPUT_DIR / zip_path.name

print("\nCopying Step6 output zip to Google Drive:")
print("from:", zip_path)
print("to:  ", drive_zip_path)

shutil.copy2(str(zip_path), str(drive_zip_path))

if zip_path.stat().st_size != drive_zip_path.stat().st_size:
    raise RuntimeError(
        "Copied Step6 zip size does not match source zip size.\n"
        f"source={zip_path.stat().st_size}\n"
        f"target={drive_zip_path.stat().st_size}"
    )

print("\nSaved Step6 output zip to Google Drive:")
print(drive_zip_path)
print("Zip size MB:", round(drive_zip_path.stat().st_size / 1024 / 1024, 2))

Creating local Step6 zip:
/content/pilot_4/data/pilot4_synthetic_disorder_lrab_2500_step6_outputs_implicit_transitive_lrab_qwen2_5_0_5b/pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe_Qwen2_5_0_5B_step6_outputs.zip
Created local zip:
/content/pilot_4/data/pilot4_synthetic_disorder_lrab_2500_step6_outputs_implicit_transitive_lrab_qwen2_5_0_5b/pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe_Qwen2_5_0_5B_step6_outputs.zip
Zip size MB: 0.08

Copying Step6 output zip to Google Drive:
from: /content/pilot_4/data/pilot4_synthetic_disorder_lrab_2500_step6_outputs_implicit_transitive_lrab_qwen2_5_0_5b/pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe_Qwen2_5_0_5B_step6_outputs.zip
to:   /content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expB_settingA1_6fps/setting_synthetic/data_synthetic_disorder_lrab_2500/step6_outputs_lrab_2500/pilot4_synthetic_disorder_lrab_2500_implicit_transitive_probe_Qwen2_5_0_5B_step6_outputs.zip

Saved Step6 output zip t

In [22]:
# ============================================================
# Optional download
# ============================================================

from google.colab import files

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>